# WiDS Global Datathon 2026 — Breakthrough Survival Stack


In [1]:
# =========================================================
# WiDS Global Datathon 2026 — Breakthrough Survival Stack
# =========================================================
import os, gc, math, random, warnings
from pathlib import Path
from collections import defaultdict

for _k in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    os.environ.setdefault(_k, '1')

import numpy as np
import pandas as pd
from scipy.special import expit, logit
from scipy.stats import norm, rankdata
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    ExtraTreesClassifier,
)

warnings.filterwarnings('ignore')

SEED = 2026
N_SPLITS = 5
N_REPEATS = 4
HORIZONS = [12, 24, 48, 72]
H2I = {h: i for i, h in enumerate(HORIZONS)}

random.seed(SEED)
np.random.seed(SEED)

HAVE_XGB = HAVE_LGB = HAVE_CB = False
try:
    import xgboost as xgb
    HAVE_XGB = True
except Exception as e:
    print('xgboost unavailable:', e)

try:
    import lightgbm as lgb
    HAVE_LGB = True
except Exception as e:
    print('lightgbm unavailable:', e)

try:
    from catboost import CatBoostClassifier
    HAVE_CB = True
except Exception as e:
    print('catboost unavailable:', e)

RAW_BASE = 'https://raw.githubusercontent.com/lequoctran181/anonymous-contest-2/refs/heads/main/'

def locate_data_dir():
    candidates = []
    for root in ['/kaggle/input', '/kaggle/working', '.', '..']:
        p = Path(root)
        if not p.exists():
            continue
        for d in p.glob('*'):
            if d.is_dir():
                names = {f.name for f in d.glob('*.csv')}
                if {'train.csv', 'test.csv', 'sample_submission.csv'}.issubset(names):
                    s = str(d).lower()
                    score = int('wids' in s) + int('global' in s) + int('dathon' in s) + int('wildfire' in s)
                    candidates.append((score, str(d)))
    if not candidates:
        return None
    candidates = sorted(candidates, key=lambda x: (-x[0], x[1]))
    return Path(candidates[0][1])

DATA_DIR = locate_data_dir()
if DATA_DIR is not None:
    train = pd.read_csv(DATA_DIR / 'train.csv')
    test = pd.read_csv(DATA_DIR / 'test.csv')
    sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
else:
    train = pd.read_csv(RAW_BASE + 'train.csv')
    test = pd.read_csv(RAW_BASE + 'test.csv')
    sample = pd.read_csv(RAW_BASE + 'sample_submission.csv')

ID_COL = 'event_id'
TIME_COL = 'time_to_hit_hours'
EVENT_COL = 'event'
BASE_FEATURES = [c for c in train.columns if c not in [ID_COL, TIME_COL, EVENT_COL]]

def signed_log1p(x):
    x = np.asarray(x, dtype=float)
    return np.sign(x) * np.log1p(np.abs(x))

def solve_eta_quadratic(dist_rem, speed, accel):
    dist_rem = np.asarray(dist_rem, dtype=float)
    speed = np.asarray(speed, dtype=float)
    accel = np.asarray(accel, dtype=float)
    out = np.full_like(dist_rem, 1e6, dtype=float)

    # dist_rem - speed * t - 0.5 * accel * t^2 = 0
    a = 0.5 * accel
    b = speed
    c = -dist_rem

    lin = np.abs(a) < 1e-9
    ok_lin = lin & (b > 1e-9)
    out[ok_lin] = dist_rem[ok_lin] / b[ok_lin]

    quad = ~lin
    disc = b[quad] ** 2 - 4 * a[quad] * c[quad]
    valid = disc >= 0
    if valid.any():
        aq = a[quad][valid]
        bq = b[quad][valid]
        cq = c[quad][valid]
        dq = np.sqrt(disc[valid])
        r1 = (-bq + dq) / (2 * aq)
        r2 = (-bq - dq) / (2 * aq)
        roots = np.vstack([r1, r2])
        roots[roots <= 1e-9] = np.nan
        rq = np.nanmin(roots, axis=0)
        idx = np.where(quad)[0][valid]
        out[idx] = np.where(np.isnan(rq), 1e6, rq)

    return np.clip(out, 0, 1e6)

def engineer_features(df):
    X = df.copy()
    eps = 1e-3

    if 'event_start_hour' in X.columns:
        X['event_start_hour_sin'] = np.sin(2 * np.pi * X['event_start_hour'] / 24.0)
        X['event_start_hour_cos'] = np.cos(2 * np.pi * X['event_start_hour'] / 24.0)
    if 'event_start_dayofweek' in X.columns:
        X['event_start_dayofweek_sin'] = np.sin(2 * np.pi * X['event_start_dayofweek'] / 7.0)
        X['event_start_dayofweek_cos'] = np.cos(2 * np.pi * X['event_start_dayofweek'] / 7.0)
    if 'event_start_month' in X.columns:
        X['event_start_month_sin'] = np.sin(2 * np.pi * (X['event_start_month'] - 1) / 12.0)
        X['event_start_month_cos'] = np.cos(2 * np.pi * (X['event_start_month'] - 1) / 12.0)

    dist = X['dist_min_ci_0_5h'].astype(float)
    close = X['closing_speed_m_per_h'].astype(float)
    along = X['along_track_speed'].astype(float)
    accel = X['dist_accel_m_per_h2'].astype(float)
    align = X['alignment_abs'].clip(0, 1).astype(float)
    radial = X['radial_growth_rate_m_per_h'].astype(float)
    proj = X['projected_advance_m'].astype(float)
    dt = X['dt_first_last_0_5h'].clip(lower=0.05).astype(float)

    dist_to5 = np.maximum(dist - 5000.0, 0.0)
    X['dist_to_5km_m'] = dist_to5
    X['dist_km'] = dist / 1000.0
    X['dist_to_5km_km'] = dist_to5 / 1000.0
    X['log1p_dist_min'] = np.log1p(dist)
    X['log1p_dist_to_5km'] = np.log1p(dist_to5)

    X['signed_log_dist_change'] = signed_log1p(X['dist_change_ci_0_5h'])
    X['signed_log_dist_slope'] = signed_log1p(X['dist_slope_ci_0_5h'])
    X['signed_log_closing_speed'] = signed_log1p(close)
    X['signed_log_along_track_speed'] = signed_log1p(along)
    X['signed_log_cross_track'] = signed_log1p(X['cross_track_component'])
    X['signed_log_dist_accel'] = signed_log1p(accel)
    X['signed_log_radial_rate'] = signed_log1p(radial)
    X['signed_log_growth_rate_ha'] = signed_log1p(X['area_growth_rate_ha_per_h'])
    X['signed_log_projected_advance'] = signed_log1p(proj)
    X['signed_log_centroid_speed'] = signed_log1p(X['centroid_speed_m_per_h'])
    X['signed_log_centroid_disp'] = signed_log1p(X['centroid_displacement_m'])
    X['signed_log_dist_std'] = signed_log1p(X['dist_std_ci_0_5h'])

    X['speed_to_dist'] = close / (dist_to5 + 250.0)
    X['along_to_dist'] = along / (dist_to5 + 250.0)
    X['radial_to_dist'] = radial / (dist_to5 + 250.0)
    X['growth_to_dist'] = X['area_growth_rate_ha_per_h'] / (X['dist_km'] + 1.0)
    X['projected_to_dist'] = proj / (dist_to5 + 250.0)
    X['uncertainty_to_dist'] = X['dist_std_ci_0_5h'] / (dist + 250.0)
    X['r2_x_closing'] = X['dist_fit_r2_0_5h'] * close

    X['align_x_closing'] = align * close
    X['align_x_along'] = X['alignment_cos'] * along
    X['align_x_radial'] = align * radial
    X['align_x_projected'] = align * proj
    X['area_x_growth'] = X['log1p_area_first'] * X['log1p_growth']
    X['area_x_close'] = X['log1p_area_first'] / (X['dist_km'] + 1.0)
    X['growth_x_close'] = X['log1p_growth'] / (X['dist_km'] + 1.0)
    X['speed_x_close'] = close / (X['dist_km'] + 1.0)
    X['radial_x_close'] = radial / (X['dist_km'] + 1.0)

    effective_close = np.maximum(0.0, along) + np.maximum(0.0, radial) * (0.35 + 0.65 * align)
    proj_rate = np.maximum(proj / dt, 0.0)

    eta_close = np.where(close > eps, dist_to5 / np.maximum(close, eps), 1e6)
    eta_eff = np.where(effective_close > eps, dist_to5 / np.maximum(effective_close, eps), 1e6)
    eta_proj = np.where(proj_rate > eps, dist_to5 / np.maximum(proj_rate, eps), 1e6)
    eta_quad = solve_eta_quadratic(dist_to5, np.maximum(close, 0.0), accel)
    eta_best = np.minimum.reduce([eta_close, eta_eff, eta_proj, eta_quad, np.full(len(X), 1e6)])

    X['eta_close_h'] = np.clip(eta_close, 0, 1e6)
    X['eta_effective_h'] = np.clip(eta_eff, 0, 1e6)
    X['eta_projected_h'] = np.clip(eta_proj, 0, 1e6)
    X['eta_quad_h'] = np.clip(eta_quad, 0, 1e6)
    X['eta_best_h'] = np.clip(eta_best, 0, 1e6)
    X['log1p_eta_best'] = np.log1p(np.minimum(eta_best, 1e4))
    X['log1p_eta_effective'] = np.log1p(np.minimum(eta_eff, 1e4))

    X['close_under_10km'] = (dist < 10000).astype(int)
    X['close_under_20km'] = (dist < 20000).astype(int)
    X['inside_5km'] = (dist <= 5000).astype(int)
    X['rapid_close_flag'] = (close > np.nanpercentile(close, 75)).astype(int)
    X['aligned_flag'] = (align > 0.7).astype(int)

    X['danger_score'] = (
        -0.85 * X['log1p_dist_to_5km']
        + 0.55 * X['signed_log_closing_speed']
        + 0.35 * X['signed_log_along_track_speed']
        + 0.25 * X['signed_log_radial_rate']
        + 0.20 * align
        + 0.18 * X['log1p_growth']
        + 0.12 * X['dist_fit_r2_0_5h']
        - 0.10 * X['low_temporal_resolution_0_5h']
    )

    for h in HORIZONS:
        X[f'heuristic_prob_{h}h'] = expit(
            (np.log1p(h) - np.log1p(np.minimum(eta_best, 1e4) + 1e-3)) / 0.85
            + 0.12 * X['danger_score']
        )

    X = X.replace([np.inf, -np.inf], np.nan)
    na_cols = [c for c in X.columns if X[c].isna().any()]
    for c in na_cols:
        X[f'{c}__na'] = X[c].isna().astype(int)
    med = X.median(numeric_only=True)
    X = X.fillna(med)

    return X

def make_horizon_target(time_arr, event_arr, horizon):
    time_arr = np.asarray(time_arr, dtype=float)
    event_arr = np.asarray(event_arr, dtype=int)
    valid = (time_arr >= horizon) | ((event_arr == 1) & (time_arr <= horizon))
    y = ((event_arr == 1) & (time_arr <= horizon)).astype(int)
    return y, valid

def harrell_c_index(time_arr, event_arr, risk):
    time_arr = np.asarray(time_arr, float)
    event_arr = np.asarray(event_arr, int)
    risk = np.asarray(risk, float)
    num = 0.0
    den = 0.0
    n = len(time_arr)
    for i in range(n):
        if event_arr[i] != 1:
            continue
        ti = time_arr[i]
        ri = risk[i]
        for j in range(n):
            if i == j:
                continue
            tj = time_arr[j]
            if ti < tj:
                den += 1.0
                rj = risk[j]
                if ri > rj:
                    num += 1.0
                elif ri == rj:
                    num += 0.5
    return num / den if den > 0 else 0.5

def hybrid_metric(time_arr, event_arr, prob_mat, rank_score=None):
    prob_mat = np.asarray(prob_mat, float)
    rank_score = prob_mat[:, 0] if rank_score is None else np.asarray(rank_score, float)
    cidx = harrell_c_index(time_arr, event_arr, rank_score)

    briers = {}
    for h in [24, 48, 72]:
        y, valid = make_horizon_target(time_arr, event_arr, h)
        if valid.sum() == 0:
            briers[h] = 0.25
        else:
            p = np.clip(prob_mat[valid, H2I[h]], 1e-6, 1 - 1e-6)
            briers[h] = np.mean((p - y[valid]) ** 2)

    wbs = 0.3 * briers[24] + 0.4 * briers[48] + 0.3 * briers[72]
    score = 0.3 * cidx + 0.7 * (1.0 - wbs)
    return score, cidx, wbs, briers

def safe_binary_prior(y):
    p = float(np.mean(y)) if len(y) else 0.20
    return float(np.clip(p, 1e-5, 1 - 1e-5))

def enforce_monotone(P):
    P = np.asarray(P, float)
    P = np.clip(P, 1e-6, 1 - 1e-6)
    P = np.maximum.accumulate(P, axis=1)
    P = np.clip(P, 1e-6, 1 - 1e-6)
    return P

def fit_hgb_binary(X_tr, y_tr, X_va, X_te, seed=42):
    base = safe_binary_prior(y_tr)
    try:
        if len(np.unique(y_tr)) < 2:
            raise ValueError('single class')
        w = np.ones(len(y_tr), dtype=float)
        pos = int((y_tr == 1).sum())
        neg = len(y_tr) - pos
        if pos > 0 and neg > 0:
            w[y_tr == 1] = math.sqrt(neg / max(pos, 1))
        model = HistGradientBoostingClassifier(
            loss='log_loss',
            learning_rate=0.04,
            max_depth=3,
            max_iter=320,
            min_samples_leaf=10,
            l2_regularization=1.0,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=35,
            random_state=seed
        )
        model.fit(X_tr, y_tr, sample_weight=w)
        return model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]
    except Exception:
        return np.full(len(X_va), base), np.full(len(X_te), base)

def fit_et_binary(X_tr, y_tr, X_va, X_te, seed=42):
    base = safe_binary_prior(y_tr)
    try:
        if len(np.unique(y_tr)) < 2:
            raise ValueError('single class')
        model = ExtraTreesClassifier(
            n_estimators=420,
            max_depth=None,
            min_samples_split=6,
            min_samples_leaf=3,
            max_features='sqrt',
            bootstrap=True,
            class_weight='balanced_subsample',
            random_state=seed,
            n_jobs=1
        )
        model.fit(X_tr, y_tr)
        return model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]
    except Exception:
        return np.full(len(X_va), base), np.full(len(X_te), base)

def fit_lgb_binary(X_tr, y_tr, X_va, X_te, seed=42):
    base = safe_binary_prior(y_tr)
    try:
        if (not HAVE_LGB) or len(np.unique(y_tr)) < 2:
            raise ValueError('lgb unavailable or single class')
        spw = max(1.0, (len(y_tr) - y_tr.sum()) / max(y_tr.sum(), 1))
        model = lgb.LGBMClassifier(
            objective='binary',
            learning_rate=0.03,
            n_estimators=900,
            num_leaves=15,
            min_child_samples=10,
            subsample=0.82,
            subsample_freq=1,
            colsample_bytree=0.78,
            reg_alpha=0.2,
            reg_lambda=2.0,
            scale_pos_weight=spw,
            random_state=seed,
            verbosity=-1,
            n_jobs=1,
        )
        model.fit(X_tr, y_tr)
        return model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]
    except Exception:
        return np.full(len(X_va), base), np.full(len(X_te), base)

def fit_xgb_binary(X_tr, y_tr, X_va, X_te, seed=42):
    base = safe_binary_prior(y_tr)
    try:
        if (not HAVE_XGB) or len(np.unique(y_tr)) < 2:
            raise ValueError('xgb unavailable or single class')
        spw = max(1.0, (len(y_tr) - y_tr.sum()) / max(y_tr.sum(), 1))
        model = xgb.XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            learning_rate=0.03,
            n_estimators=900,
            max_depth=3,
            min_child_weight=3,
            subsample=0.85,
            colsample_bytree=0.72,
            reg_alpha=0.10,
            reg_lambda=2.0,
            gamma=0.0,
            max_bin=256,
            tree_method='hist',
            scale_pos_weight=spw,
            random_state=seed,
            n_jobs=1,
            verbosity=0,
        )
        model.fit(X_tr, y_tr)
        return model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]
    except Exception:
        return np.full(len(X_va), base), np.full(len(X_te), base)

def fit_cb_binary(X_tr, y_tr, X_va, X_te, seed=42):
    base = safe_binary_prior(y_tr)
    try:
        if (not HAVE_CB) or len(np.unique(y_tr)) < 2:
            raise ValueError('catboost unavailable or single class')
        pos = int((y_tr == 1).sum())
        neg = len(y_tr) - pos
        model = CatBoostClassifier(
            loss_function='Logloss',
            eval_metric='Logloss',
            learning_rate=0.03,
            iterations=900,
            depth=5,
            l2_leaf_reg=10.0,
            random_strength=0.5,
            bootstrap_type='Bayesian',
            bagging_temperature=1.0,
            grow_policy='SymmetricTree',
            class_weights=[1.0, max(1.0, neg / max(pos, 1))],
            random_seed=seed,
            allow_writing_files=False,
            verbose=False,
        )
        model.fit(X_tr, y_tr)
        return model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]
    except Exception:
        return np.full(len(X_va), base), np.full(len(X_te), base)

def with_horizon_cols(df, h):
    out = df.copy()
    out['__horizon_h__'] = float(h)
    out['__horizon_log__'] = float(np.log(h))
    out['__horizon_inv__'] = 1.0 / float(h)
    out['__is_h12__'] = float(h == 12)
    out['__is_h24__'] = float(h == 24)
    out['__is_h48__'] = float(h == 48)
    out['__is_h72__'] = float(h == 72)
    return out

POOL_H_WEIGHTS = {12: 0.18, 24: 0.30, 48: 0.40, 72: 0.30}

def build_pooled_dataset(X_df, time_arr, event_arr):
    frames, ys, ws = [], [], []
    for h in HORIZONS:
        y_h, valid_h = make_horizon_target(time_arr, event_arr, h)
        X_h = with_horizon_cols(X_df.iloc[np.where(valid_h)[0]].reset_index(drop=True), h)
        frames.append(X_h)
        ys.append(y_h[valid_h])
        pos = int(y_h[valid_h].sum())
        neg = int(valid_h.sum() - pos)
        w = np.full(valid_h.sum(), POOL_H_WEIGHTS[h], dtype=float)
        if pos > 0 and neg > 0:
            w[y_h[valid_h] == 1] *= math.sqrt(neg / max(pos, 1))
        ws.append(w)
    X_pool = pd.concat(frames, axis=0, ignore_index=True)
    y_pool = np.concatenate(ys)
    w_pool = np.concatenate(ws)
    return X_pool, y_pool, w_pool

def fit_pooled_hgb(X_df_tr, time_tr, event_tr, X_df_va, X_df_te, seed=42):
    try:
        X_pool, y_pool, w_pool = build_pooled_dataset(X_df_tr, time_tr, event_tr)
        base = safe_binary_prior(y_pool)
        if len(np.unique(y_pool)) < 2:
            raise ValueError('single class pooled')
        model = HistGradientBoostingClassifier(
            loss='log_loss',
            learning_rate=0.035,
            max_depth=3,
            max_iter=280,
            min_samples_leaf=12,
            l2_regularization=1.0,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=35,
            random_state=seed
        )
        model.fit(X_pool.values.astype(np.float32), y_pool, sample_weight=w_pool)
        va_mat = np.zeros((len(X_df_va), len(HORIZONS)), dtype=float)
        te_mat = np.zeros((len(X_df_te), len(HORIZONS)), dtype=float)
        for j, h in enumerate(HORIZONS):
            va_mat[:, j] = model.predict_proba(with_horizon_cols(X_df_va, h).values.astype(np.float32))[:, 1]
            te_mat[:, j] = model.predict_proba(with_horizon_cols(X_df_te, h).values.astype(np.float32))[:, 1]
        return va_mat, te_mat
    except Exception:
        base = 0.20
        return np.full((len(X_df_va), len(HORIZONS)), base), np.full((len(X_df_te), len(HORIZONS)), base)

def make_dmatrix_aft(X_in, time_in, event_in):
    dmat = xgb.DMatrix(X_in)
    lower = np.asarray(time_in, dtype=float)
    upper = lower.copy()
    upper[np.asarray(event_in, dtype=int) == 0] = np.inf
    dmat.set_float_info('label_lower_bound', lower)
    dmat.set_float_info('label_upper_bound', upper)
    return dmat

def aft_probs_from_mu(mu, dist_name='normal', scale=1.0):
    h_arr = np.array(HORIZONS, dtype=float)
    log_h = np.log(h_arr)[None, :]
    mu = np.asarray(mu, dtype=float)[:, None]
    if dist_name == 'normal':
        probs = norm.cdf((log_h - mu) / scale)
    elif dist_name == 'logistic':
        probs = expit((log_h - mu) / scale)
    else:
        probs = expit((log_h - mu) / scale)
    return np.clip(probs, 1e-6, 1 - 1e-6)

def fit_xgb_aft_probs(X_tr, time_tr, event_tr, X_va, X_te, dist_name='normal', scale=1.0, seed=42):
    base_mat = np.tile(np.array([0.05, 0.10, 0.20, 0.30]), (len(X_va), 1))
    base_test = np.tile(np.array([0.05, 0.10, 0.20, 0.30]), (len(X_te), 1))
    raw_base_va = np.full(len(X_va), np.log(48.0), dtype=float)
    raw_base_te = np.full(len(X_te), np.log(48.0), dtype=float)
    try:
        if not HAVE_XGB:
            raise ValueError('xgb unavailable')
        dtrain = make_dmatrix_aft(X_tr, time_tr, event_tr)
        params = {
            'objective': 'survival:aft',
            'eval_metric': 'aft-nloglik',
            'tree_method': 'hist',
            'learning_rate': 0.03,
            'max_depth': 3,
            'min_child_weight': 4,
            'subsample': 0.85,
            'colsample_bytree': 0.75,
            'lambda': 2.0,
            'alpha': 0.1,
            'max_bin': 256,
            'aft_loss_distribution': dist_name,
            'aft_loss_distribution_scale': scale,
            'seed': seed,
            'verbosity': 0,
        }
        booster = xgb.train(params, dtrain, num_boost_round=900)
        mu_va = booster.predict(xgb.DMatrix(X_va))
        mu_te = booster.predict(xgb.DMatrix(X_te))
        return aft_probs_from_mu(mu_va, dist_name, scale), aft_probs_from_mu(mu_te, dist_name, scale), mu_va, mu_te
    except Exception:
        return base_mat, base_test, raw_base_va, raw_base_te

def fit_time_reg_hgb(X_tr, time_tr, event_tr, X_va, X_te, seed=42):
    try:
        hit_idx = np.where(np.asarray(event_tr) == 1)[0]
        if len(hit_idx) < 8:
            raise ValueError('too few hits')
        X_hit = X_tr[hit_idx]
        y_hit = np.log1p(np.clip(np.asarray(time_tr)[hit_idx], 0, None))
        model = HistGradientBoostingRegressor(
            learning_rate=0.035,
            max_depth=3,
            max_iter=320,
            min_samples_leaf=8,
            l2_regularization=0.5,
            early_stopping=True,
            validation_fraction=0.18,
            n_iter_no_change=40,
            random_state=seed
        )
        model.fit(X_hit, y_hit)
        return model.predict(X_va), model.predict(X_te)
    except Exception:
        base = float(np.log1p(np.median(time_tr[event_tr == 1])) if (event_tr == 1).any() else np.log1p(48.0))
        return np.full(len(X_va), base), np.full(len(X_te), base)

all_raw = pd.concat([train[BASE_FEATURES], test[BASE_FEATURES]], axis=0, ignore_index=True)
all_eng = engineer_features(all_raw)
X_train_df = all_eng.iloc[:len(train)].reset_index(drop=True).copy()
X_test_df = all_eng.iloc[len(train):].reset_index(drop=True).copy()

X = X_train_df.values.astype(np.float32)
X_test = X_test_df.values.astype(np.float32)
times = train[TIME_COL].values.astype(float)
events = train[EVENT_COL].values.astype(int)

n_train = len(train)
n_test = len(test)

global_prior_vec = []
for h in HORIZONS:
    y_h, valid_h = make_horizon_target(times, events, h)
    p = y_h[valid_h].mean() if valid_h.any() else events.mean()
    global_prior_vec.append(float(np.clip(p, 1e-5, 1 - 1e-5)))
global_prior_vec = np.array(global_prior_vec, dtype=float)

dist_col = 'dist_min_ci_0_5h' if 'dist_min_ci_0_5h' in X_train_df.columns else 'dist_to_5km_m'
q1, q2 = np.quantile(X_train_df[dist_col].values, [0.35, 0.70])
train_regime = np.digitize(X_train_df[dist_col].values, [q1, q2])
test_regime = np.digitize(X_test_df[dist_col].values, [q1, q2])

regime_prior_train = np.zeros((n_train, len(HORIZONS)), dtype=float)
regime_prior_test = np.zeros((n_test, len(HORIZONS)), dtype=float)
for r in range(3):
    train_mask_r = train_regime == r
    test_mask_r = test_regime == r
    for j, h in enumerate(HORIZONS):
        y_h, valid_h = make_horizon_target(times, events, h)
        mask = train_mask_r & valid_h
        if mask.any():
            p = float(np.clip(y_h[mask].mean(), 1e-5, 1 - 1e-5))
        else:
            p = global_prior_vec[j]
        regime_prior_train[train_mask_r, j] = p
        regime_prior_test[test_mask_r, j] = p

prior_train_mat = 0.55 * np.tile(global_prior_vec, (n_train, 1)) + 0.45 * regime_prior_train
prior_test_mat = 0.55 * np.tile(global_prior_vec, (n_test, 1)) + 0.45 * regime_prior_test

prob_store = {}
raw_store = {}

def update_prob_store(name, va_idx, va_mat, te_mat):
    if name not in prob_store:
        prob_store[name] = {
            'oof_sum': np.zeros((n_train, len(HORIZONS)), dtype=float),
            'oof_cnt': np.zeros((n_train, len(HORIZONS)), dtype=float),
            'test_sum': np.zeros((n_test, len(HORIZONS)), dtype=float),
            'test_cnt': np.zeros((n_test, len(HORIZONS)), dtype=float),
        }
    prob_store[name]['oof_sum'][va_idx] += va_mat
    prob_store[name]['oof_cnt'][va_idx] += 1.0
    prob_store[name]['test_sum'] += te_mat
    prob_store[name]['test_cnt'] += 1.0

def update_raw_store(name, va_idx, va_pred, te_pred):
    if name not in raw_store:
        raw_store[name] = {
            'oof_sum': np.zeros(n_train, dtype=float),
            'oof_cnt': np.zeros(n_train, dtype=float),
            'test_sum': np.zeros(n_test, dtype=float),
            'test_cnt': np.zeros(n_test, dtype=float),
        }
    raw_store[name]['oof_sum'][va_idx] += va_pred
    raw_store[name]['oof_cnt'][va_idx] += 1.0
    raw_store[name]['test_sum'] += te_pred
    raw_store[name]['test_cnt'] += 1.0

direct_models = [
    ('hgb_bin', fit_hgb_binary),
    ('et_bin', fit_et_binary),
]
if HAVE_LGB:
    direct_models.append(('lgb_bin', fit_lgb_binary))
if HAVE_XGB:
    direct_models.append(('xgb_bin', fit_xgb_binary))
if HAVE_CB:
    direct_models.append(('cb_bin', fit_cb_binary))

splitter = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)

for fold, (tr_idx, va_idx) in enumerate(splitter.split(X, events), 1):
    print(f'fold {fold:02d}/{N_SPLITS * N_REPEATS}')
    X_tr, X_va = X[tr_idx], X[va_idx]
    t_tr, e_tr = times[tr_idx], events[tr_idx]

    X_df_tr = X_train_df.iloc[tr_idx].reset_index(drop=True)
    X_df_va = X_train_df.iloc[va_idx].reset_index(drop=True)

    seed_fold = SEED + fold * 37

    for fname, ffn in direct_models:
        va_mat = np.zeros((len(va_idx), len(HORIZONS)), dtype=float)
        te_mat = np.zeros((n_test, len(HORIZONS)), dtype=float)
        for j, h in enumerate(HORIZONS):
            y_tr_h, valid_tr_h = make_horizon_target(t_tr, e_tr, h)
            if valid_tr_h.sum() < 16 or y_tr_h[valid_tr_h].sum() < 2:
                base = global_prior_vec[j]
                va_pred = np.full(len(va_idx), base)
                te_pred = np.full(n_test, base)
            else:
                va_pred, te_pred = ffn(
                    X_tr[valid_tr_h],
                    y_tr_h[valid_tr_h],
                    X_va,
                    X_test,
                    seed=seed_fold + 101 * (j + 1)
                )
            va_mat[:, j] = np.clip(va_pred, 1e-5, 1 - 1e-5)
            te_mat[:, j] = np.clip(te_pred, 1e-5, 1 - 1e-5)
        update_prob_store(fname, va_idx, va_mat, te_mat)

    va_pool, te_pool = fit_pooled_hgb(X_df_tr, t_tr, e_tr, X_df_va, X_test_df, seed=seed_fold + 911)
    update_prob_store('pooled_hgb', va_idx, np.clip(va_pool, 1e-5, 1 - 1e-5), np.clip(te_pool, 1e-5, 1 - 1e-5))

    if HAVE_XGB:
        va_aft_n, te_aft_n, mu_va_n, mu_te_n = fit_xgb_aft_probs(
            X_tr, t_tr, e_tr, X_va, X_test, dist_name='normal', scale=1.00, seed=seed_fold + 1201
        )
        update_prob_store('aft_normal', va_idx, va_aft_n, te_aft_n)
        update_raw_store('aft_normal_mu', va_idx, mu_va_n, mu_te_n)

        va_aft_l, te_aft_l, mu_va_l, mu_te_l = fit_xgb_aft_probs(
            X_tr, t_tr, e_tr, X_va, X_test, dist_name='logistic', scale=1.20, seed=seed_fold + 1202
        )
        update_prob_store('aft_logistic', va_idx, va_aft_l, te_aft_l)
        update_raw_store('aft_logistic_mu', va_idx, mu_va_l, mu_te_l)

    va_eta, te_eta = fit_time_reg_hgb(X_tr, t_tr, e_tr, X_va, X_test, seed=seed_fold + 1501)
    update_raw_store('eta_reg_hgb', va_idx, va_eta, te_eta)

    gc.collect()

prob_models = {}
for name, d in prob_store.items():
    oof = d['oof_sum'] / np.maximum(d['oof_cnt'], 1.0)
    te = d['test_sum'] / np.maximum(d['test_cnt'], 1.0)
    oof = np.where(np.isfinite(oof), oof, prior_train_mat)
    te = np.where(np.isfinite(te), te, prior_test_mat)
    oof = enforce_monotone(np.clip(oof, 1e-5, 1 - 1e-5))
    te = enforce_monotone(np.clip(te, 1e-5, 1 - 1e-5))
    prob_models[name] = {'oof': oof, 'test': te}

raw_models = {}
for name, d in raw_store.items():
    oof = d['oof_sum'] / np.maximum(d['oof_cnt'], 1.0)
    te = d['test_sum'] / np.maximum(d['test_cnt'], 1.0)
    if not np.isfinite(oof).all():
        fill = np.nanmedian(oof[np.isfinite(oof)]) if np.isfinite(oof).any() else 0.0
        oof = np.where(np.isfinite(oof), oof, fill)
    if not np.isfinite(te).all():
        fill = np.nanmedian(te[np.isfinite(te)]) if np.isfinite(te).any() else 0.0
        te = np.where(np.isfinite(te), te, fill)
    raw_models[name] = {'oof': oof, 'test': te}

rows = []
for name, obj in prob_models.items():
    sc, cidx, wbs, br = hybrid_metric(times, events, obj['oof'])
    rows.append({
        'model': name,
        'hybrid': sc,
        'cindex': cidx,
        'wbs': wbs,
        'brier24': br[24],
        'brier48': br[48],
        'brier72': br[72],
    })

scores_df = pd.DataFrame(rows).sort_values(['hybrid', 'cindex'], ascending=[False, False]).reset_index(drop=True)
print(scores_df)

score_map = scores_df.set_index('model')['hybrid'].to_dict()
brier24_map = scores_df.set_index('model')['brier24'].to_dict()
brier48_map = scores_df.set_index('model')['brier48'].to_dict()
brier72_map = scores_df.set_index('model')['brier72'].to_dict()

selected = []
family_counts = defaultdict(int)
for name in scores_df['model'].tolist():
    fam = name.split('_')[0]
    if family_counts[fam] >= 2:
        continue
    selected.append(name)
    family_counts[fam] += 1
    if len(selected) >= min(8, len(scores_df)):
        break

if len(selected) < min(4, len(scores_df)):
    selected = scores_df['model'].head(min(4, len(scores_df))).tolist()

print('selected models:', selected)

def weighted_mean_candidate(model_names):
    tr = np.zeros((n_train, len(HORIZONS)), dtype=float)
    te = np.zeros((n_test, len(HORIZONS)), dtype=float)
    for j, h in enumerate(HORIZONS):
        if h == 12:
            ws = np.array([max(score_map[nm] - 0.5, 1e-6) for nm in model_names], dtype=float)
        elif h == 24:
            ws = np.array([1.0 / max(brier24_map[nm], 1e-6) for nm in model_names], dtype=float)
        elif h == 48:
            ws = np.array([1.0 / max(brier48_map[nm], 1e-6) for nm in model_names], dtype=float)
        else:
            ws = np.array([1.0 / max(brier72_map[nm], 1e-6) for nm in model_names], dtype=float)
        ws = ws / ws.sum()
        for w, nm in zip(ws, model_names):
            tr[:, j] += w * prob_models[nm]['oof'][:, j]
            te[:, j] += w * prob_models[nm]['test'][:, j]
    return tr, te

simple_mean_train = np.mean([prob_models[nm]['oof'] for nm in selected], axis=0)
simple_mean_test = np.mean([prob_models[nm]['test'] for nm in selected], axis=0)
weighted_train, weighted_test = weighted_mean_candidate(selected)

aux_train = pd.DataFrame(index=np.arange(n_train))
aux_test = pd.DataFrame(index=np.arange(n_test))

base_aux_cols = []
for c in [
    'danger_score',
    'log1p_dist_to_5km',
    'signed_log_closing_speed',
    'signed_log_along_track_speed',
    'signed_log_radial_rate',
    'log1p_eta_best',
    'low_temporal_resolution_0_5h',
    'dist_fit_r2_0_5h',
    'dist_km',
    'close_under_10km',
    'close_under_20km',
]:
    if c in X_train_df.columns:
        aux_train[c] = X_train_df[c].values
        aux_test[c] = X_test_df[c].values
        base_aux_cols.append(c)

for h in HORIZONS:
    c = f'heuristic_prob_{h}h'
    if c in X_train_df.columns:
        aux_train[c] = X_train_df[c].values
        aux_test[c] = X_test_df[c].values

for nm, obj in raw_models.items():
    aux_train[nm] = obj['oof']
    aux_test[nm] = obj['test']
    base_aux_cols.append(nm)

for r in range(3):
    aux_train[f'regime_{r}'] = (train_regime == r).astype(float)
    aux_test[f'regime_{r}'] = (test_regime == r).astype(float)
    base_aux_cols.append(f'regime_{r}')

selected_prob_train = {nm: prob_models[nm]['oof'] for nm in selected}
selected_prob_test = {nm: prob_models[nm]['test'] for nm in selected}

def build_meta_matrix(prob_dict, aux_df, h_idx, model_names):
    cols = []
    for nm in model_names:
        p = np.clip(prob_dict[nm][:, h_idx], 1e-5, 1 - 1e-5)
        cols.append(logit(p))
    for c in base_aux_cols:
        cols.append(aux_df[c].values)
    heur_col = f'heuristic_prob_{HORIZONS[h_idx]}h'
    if heur_col in aux_df.columns:
        cols.append(aux_df[heur_col].values)
    return np.column_stack(cols).astype(np.float32)

def fit_meta_stacks():
    lr_tr = np.zeros((n_train, len(HORIZONS)), dtype=float)
    lr_te = np.zeros((n_test, len(HORIZONS)), dtype=float)
    rd_tr = np.zeros((n_train, len(HORIZONS)), dtype=float)
    rd_te = np.zeros((n_test, len(HORIZONS)), dtype=float)

    for j, h in enumerate(HORIZONS):
        y_h, valid_h = make_horizon_target(times, events, h)
        Mtr = build_meta_matrix(selected_prob_train, aux_train, j, selected)
        Mte = build_meta_matrix(selected_prob_test, aux_test, j, selected)

        if valid_h.sum() < 20 or y_h[valid_h].sum() < 2 or y_h[valid_h].sum() == valid_h.sum():
            base = global_prior_vec[j]
            lr_tr[:, j] = base
            lr_te[:, j] = base
            rd_tr[:, j] = base
            rd_te[:, j] = base
            continue

        try:
            cw = 'balanced' if h == 12 else None
            lr = make_pipeline(
                StandardScaler(),
                LogisticRegression(C=0.35, solver='liblinear', class_weight=cw, max_iter=4000)
            )
            lr.fit(Mtr[valid_h], y_h[valid_h])
            lr_tr[:, j] = lr.predict_proba(Mtr)[:, 1]
            lr_te[:, j] = lr.predict_proba(Mte)[:, 1]
        except Exception:
            base = global_prior_vec[j]
            lr_tr[:, j] = base
            lr_te[:, j] = base

        try:
            rg = make_pipeline(StandardScaler(), Ridge(alpha=4.0))
            rg.fit(Mtr[valid_h], y_h[valid_h])
            rd_tr[:, j] = np.clip(rg.predict(Mtr), 1e-5, 1 - 1e-5)
            rd_te[:, j] = np.clip(rg.predict(Mte), 1e-5, 1 - 1e-5)
        except Exception:
            base = global_prior_vec[j]
            rd_tr[:, j] = base
            rd_te[:, j] = base

    return lr_tr, lr_te, rd_tr, rd_te

lr_train, lr_test, ridge_train, ridge_test = fit_meta_stacks()

def shrink_to_prior(P_train, P_test):
    best = None
    for alpha in [0.90, 0.94, 0.97, 1.00]:
        tr = enforce_monotone(alpha * P_train + (1.0 - alpha) * prior_train_mat)
        te = enforce_monotone(alpha * P_test + (1.0 - alpha) * prior_test_mat)
        sc, cidx, wbs, br = hybrid_metric(times, events, tr)
        row = (sc, tr, te, alpha, cidx, wbs, br)
        if (best is None) or (row[0] > best[0]):
            best = row
    return best

candidate_inputs = {
    'simple_mean': (simple_mean_train, simple_mean_test),
    'weighted_mean': (weighted_train, weighted_test),
    'lr_stack': (lr_train, lr_test),
    'ridge_stack': (ridge_train, ridge_test),
}

candidate_results = {name: shrink_to_prior(tr, te) for name, (tr, te) in candidate_inputs.items()}

cand_rows = []
for name, res in candidate_results.items():
    sc, _, _, alpha, cidx, wbs, br = res
    cand_rows.append({
        'candidate': name,
        'hybrid': sc,
        'alpha': alpha,
        'cindex': cidx,
        'wbs': wbs,
        'brier24': br[24],
        'brier48': br[48],
        'brier72': br[72],
    })

cand_df = pd.DataFrame(cand_rows).sort_values(['hybrid', 'cindex'], ascending=[False, False]).reset_index(drop=True)
print(cand_df)

best_name = cand_df.iloc[0]['candidate']
final_train = candidate_results[best_name][1].copy()
final_test = candidate_results[best_name][2].copy()

def empirical_quantile_map(reference_values, u):
    reference_values = np.sort(np.asarray(reference_values, dtype=float))
    if len(reference_values) == 1:
        return np.full_like(u, reference_values[0], dtype=float)
    grid = np.linspace(0.0, 1.0, len(reference_values))
    return np.interp(np.clip(u, 0.0, 1.0), grid, reference_values)

def unit_rank(x):
    x = np.asarray(x, dtype=float)
    return (rankdata(x, method='average') - 0.5) / len(x)

rank_train_components = []
rank_test_components = []

for nm in selected:
    rank_train_components.append(prob_models[nm]['oof'][:, 0])
    rank_test_components.append(prob_models[nm]['test'][:, 0])

for nm in ['aft_normal_mu', 'aft_logistic_mu', 'eta_reg_hgb']:
    if nm in raw_models:
        rank_train_components.append(-raw_models[nm]['oof'])
        rank_test_components.append(-raw_models[nm]['test'])

if 'danger_score' in aux_train.columns:
    rank_train_components.append(aux_train['danger_score'].values)
    rank_test_components.append(aux_test['danger_score'].values)

if 'heuristic_prob_12h' in aux_train.columns:
    rank_train_components.append(aux_train['heuristic_prob_12h'].values)
    rank_test_components.append(aux_test['heuristic_prob_12h'].values)

u_train = np.mean(np.column_stack([unit_rank(c) for c in rank_train_components]), axis=1)
u_test = np.mean(np.column_stack([unit_rank(c) for c in rank_test_components]), axis=1)

base12_train = final_train[:, 0].copy()
base12_test = final_test[:, 0].copy()

reorder_train = empirical_quantile_map(base12_train, u_train)
reorder_test = empirical_quantile_map(base12_train, u_test)

best_rank = None
for gamma in [1.00, 0.85, 0.70, 0.55, 0.40, 0.25]:
    tr = final_train.copy()
    te = final_test.copy()
    tr[:, 0] = gamma * base12_train + (1.0 - gamma) * reorder_train
    te[:, 0] = gamma * base12_test + (1.0 - gamma) * reorder_test
    tr[:, 0] = np.minimum(tr[:, 0], np.maximum(1e-6, tr[:, 1] - 1e-6))
    te[:, 0] = np.minimum(te[:, 0], np.maximum(1e-6, te[:, 1] - 1e-6))
    tr = enforce_monotone(tr)
    te = enforce_monotone(te)
    sc, cidx, wbs, br = hybrid_metric(times, events, tr)
    row = (sc, tr, te, gamma, cidx, wbs, br)
    if (best_rank is None) or (row[0] > best_rank[0]):
        best_rank = row

final_train = best_rank[1]
final_test = best_rank[2]

print({
    'best_candidate': best_name,
    'best_candidate_alpha': candidate_results[best_name][3],
    'rank_gamma': best_rank[3],
    'local_hybrid': best_rank[0],
    'local_cindex': best_rank[4],
    'local_wbs': best_rank[5],
})

pred_df = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    'prob_12h': final_test[:, 0],
    'prob_24h': final_test[:, 1],
    'prob_48h': final_test[:, 2],
    'prob_72h': final_test[:, 3],
})

pred_df[['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']] = np.clip(
    pred_df[['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']].values, 0.0, 1.0
)
pred_df[['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']] = np.maximum.accumulate(
    pred_df[['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']].values, axis=1
)

submission = sample[[ID_COL]].merge(pred_df, on=ID_COL, how='left')

assert submission[ID_COL].is_unique
assert set(submission[ID_COL]) == set(test[ID_COL])
arr = submission[['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']].values
assert np.isfinite(arr).all()
assert ((arr >= 0.0) & (arr <= 1.0)).all()
assert np.all(arr[:, 0] <= arr[:, 1] + 1e-12)
assert np.all(arr[:, 1] <= arr[:, 2] + 1e-12)
assert np.all(arr[:, 2] <= arr[:, 3] + 1e-12)

submission.to_csv('submission.csv', index=False)
submission.head()


fold 01/20
fold 02/20
fold 03/20
fold 04/20
fold 05/20
fold 06/20
fold 07/20
fold 08/20
fold 09/20
fold 10/20
fold 11/20
fold 12/20
fold 13/20
fold 14/20
fold 15/20
fold 16/20
fold 17/20
fold 18/20
fold 19/20
fold 20/20
          model    hybrid    cindex       wbs   brier24   brier48  \
0    pooled_hgb  0.972166  0.941123  0.014530  0.027620  0.014850   
1       hgb_bin  0.971074  0.936320  0.014032  0.025434  0.016003   
2       lgb_bin  0.970129  0.934250  0.014495  0.027585  0.015548   
3        cb_bin  0.968921  0.936982  0.017391  0.032078  0.019418   
4       xgb_bin  0.967562  0.938556  0.020007  0.035101  0.023693   
5        et_bin  0.967024  0.936237  0.019781  0.033701  0.024178   
6    aft_normal  0.841610  0.879389  0.174581  0.125981  0.116956   
7  aft_logistic  0.804660  0.897110  0.234961  0.158273  0.161984   

        brier72  
0  1.012573e-03  
1  1.000000e-10  
2  1.000000e-10  
3  1.000000e-10  
4  1.000000e-10  
5  1.000000e-10  
6  3.000140e-01  
7  4.089520e-0

,event_id,prob_12h,prob_24h,prob_48h,prob_72h
0,10662602,0.000010,0.005453,0.014272,0.99999
1,13353600,0.587875,0.939212,0.999990,0.99999
2,13942327,0.045713,0.045714,0.045714,0.99999
3,16112781,0.591860,0.919006,0.999990,0.99999
4,17132808,0.035282,0.041083,0.041083,0.99999
